# 04 · June agent work list and model package
Scores the June cohort with the calibrated model, removes dispute and hardship accounts, ranks by **expected unpaid amount = (1 − p) × amount_due**, cuts at agent capacity (**artifact 8**) and packages the model (**final output 3**).

In [ ]:
import json

import joblib
import pandas as pd

import pipeline_config as cfg

df = pd.read_csv(cfg.CLEAN_FILE)
june = df[df["cohort_month"] == cfg.TEST_MONTH].copy()
features = cfg.feature_columns(df)
model = joblib.load(cfg.MODEL_DIR / "gradient_boosting.joblib")
calibrator = joblib.load(cfg.MODEL_DIR / "calibrator.joblib")
print(f"June cohort: {len(june):,} accounts")

## 1. Score and rank

In [ ]:
june["p_pay"] = calibrator.predict(model.predict_proba(june[features])[:, 1])
june["expected_unpaid"] = (1 - june["p_pay"]) * june["amount_due"]

capacity = round(cfg.AGENT_CAPACITY_SHARE * len(june))
specialist = (june[cfg.SPECIALIST_FLAGS] == 1).any(axis=1)
print(f"agent capacity {capacity:,} | routed to specialist team (dispute / hardship): {specialist.sum():,}")

work_list = (june[~specialist].nlargest(capacity, "expected_unpaid")
             .assign(priority=lambda d: range(1, len(d) + 1))
             [["priority", "account_id", "product_type", "days_past_due", "amount_due", "p_pay",
               "expected_unpaid", "preferred_channel", "has_open_dispute", "hardship_flag", cfg.TARGET]]
             .rename(columns={cfg.TARGET: "actual_paid"}))
work_list.to_csv(cfg.WORKLIST_FILE, index=False)
print(f"saved {len(work_list):,} accounts -> {cfg.WORKLIST_FILE.name}")
work_list.head()

## 2. What the list achieves (June outcomes are known)

In [ ]:
eligible = june[~specialist]
actual_unpaid = (1 - eligible[cfg.TARGET]) * eligible["amount_due"]
dpd_list = eligible.nlargest(capacity, "days_past_due")

result = {
    "june_accounts": len(june),
    "agent_capacity": capacity,
    "expected_unpaid_on_list_usd": round(float(work_list["expected_unpaid"].sum())),
    "actual_unpaid_on_list_usd": round(float(((1 - work_list["actual_paid"]) * work_list["amount_due"]).sum())),
    "total_unpaid_eligible_usd": round(float(actual_unpaid.sum())),
    "coverage_model": round(float(((1 - work_list["actual_paid"]) * work_list["amount_due"]).sum() / actual_unpaid.sum()), 3),
    "coverage_dpd_rule": round(float(((1 - dpd_list[cfg.TARGET]) * dpd_list["amount_due"]).sum() / actual_unpaid.sum()), 3),
}
pd.Series(result)

## 3. Package the model

In [ ]:
cfg.PACKAGE_DIR.mkdir(exist_ok=True)
joblib.dump(model, cfg.PACKAGE_DIR / "model.joblib")
joblib.dump(calibrator, cfg.PACKAGE_DIR / "calibrator.joblib")

comparison = pd.read_csv(cfg.COMPARISON_FILE, index_col="model")
metrics = {
    "june_gradient_boosting": comparison.loc["gradient_boosting"].dropna().round(4).to_dict(),
    "june_baseline_pr_auc": round(float(comparison.loc["baseline_logreg", "pr_auc"]), 4),
    "work_list": result,
    "features": features,
    "excluded_columns": cfg.EXCLUDED,
    "windows": {"train": cfg.TRAIN_MONTHS, "calibrate": cfg.CALIBRATION_MONTH, "test": cfg.TEST_MONTH},
}
(cfg.PACKAGE_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("saved model.joblib, calibrator.joblib and metrics.json")

In [ ]:
g = metrics["june_gradient_boosting"]
readme = f"""# Collections propensity-to-pay model

Predicts the probability that an account in early collections pays the past-due amount in full within
30 days. Scored on **day {cfg.SCORING_DAY}** after the account enters collections.
Trained on January-April 2026, calibrated on May, tested out of time on June.

## How to build the monthly agent list
1. Run `2_Code/01_data_preprocessing.ipynb` on the new month (same cleaning rules).
2. `p = calibrator.predict(model.predict_proba(cleaned[features])[:, 1])` (feature list in `metrics.json`).
3. `expected_unpaid = (1 - p) * amount_due`; route dispute / hardship accounts to the specialist team.
4. Give agents the top {cfg.AGENT_CAPACITY_SHARE:.0%} of the cohort by expected unpaid; nudge the rest by SMS / app.

## Excluded columns
- `account_status_at_extract`, `amount_recovered_30d`, `total_contact_attempts_30d`: captured after day {cfg.SCORING_DAY} (leakage)
- IDs, dates and the cohort month

## Performance (June, out of time)
| Check | Result |
|---|---|
| ROC-AUC | {g["roc_auc"]:.3f} |
| PR-AUC | {g["pr_auc"]:.3f} (baseline {metrics["june_baseline_pr_auc"]:.3f}) |
| Brier raw / calibrated | {g["brier"]:.4f} / {g["brier_calibrated"]:.4f} |
| Unpaid amount covered by the agent list | {result["coverage_model"]:.0%} vs {result["coverage_dpd_rule"]:.0%} with highest-DPD-first |

## Limitations
- Pay rates fell every month in 2026 (BNPL fastest): re-calibrate monthly, retrain quarterly.
- Probabilities describe paying *with* the current treatment mix; they are not treatment effects.
"""
(cfg.PACKAGE_DIR / "README.md").write_text(readme, encoding="utf-8")
print(readme)